# 3.2 — Segmentation des phrases (TP à compléter)

**Traitement automatique du langage naturel (TALN) | Python, spaCy et NLTK**

Ce notebook présente plusieurs méthodes pour découper un texte en phrases, examiner les frontières entre phrases et adapter la segmentation à des besoins particuliers.

## Objectifs pédagogiques

À la fin de ce notebook, vous serez capable de :
- expliquer la segmentation des phrases ;
- utiliser `spaCy` et `NLTK` pour segmenter un texte ;
- inspecter les frontières de phrases dans `spaCy` ;
- comprendre pourquoi la ponctuation seule ne suffit pas toujours ;
- tester la segmentation sur des textes anglais et arabes.

## Consignes de travail

Ce notebook est un TP guidé à compléter progressivement.

- Remplacez chaque `...` par l'instruction ou l'expression correcte.
- Exécutez les cellules dans l'ordre et observez les résultats.
- Avant d'exécuter une cellule, prédisez ce qu'elle va afficher.
- En cas d'erreur, lisez le message et corrigez votre réponse à partir du cours.
- Complétez le bilan individuel à la fin.

**Conseil :** ne remplacez pas les `...` au hasard. Appuyez-vous sur les explications et sur les résultats des cellules précédentes.

## 1. Préparation de l’environnement

Exécutez cette cellule si les bibliothèques ne sont pas encore installées. Dans Google Colab, vous pouvez l’exécuter directement.

In [ ]:
%pip install -q spacy nltk

## 2. Charger le modèle anglais de spaCy

Le modèle `en_core_web_sm` fournit un analyseur syntaxique qui peut déterminer les frontières entre phrases. Si le modèle n’est pas installé, la cellule suivante le télécharge.

In [1]:
import spacy

try:
    nlp = spacy.load("en_core_web_sm")
except OSError:
    from spacy.cli import download
    download("en_core_web_sm")
    nlp = spacy.load("en_core_web_sm")

print("Pipeline:", nlp.pipe_names)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 121.6 kB/s  0:01:330:00:0200:05



[notice] A new release of pip is available: 26.0 -> 26.2.1
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip


✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
Pipeline: ['tok2vec', 'tagger', 'parser', 'attribute_ruler', 'lemmatizer', 'ner']


## 3. Segmenter un texte avec spaCy

In [3]:
# EXERCICE 1 — Segmenter un texte avec spaCy
text = (
    "This is the first sentence. "
    "This is another sentence. "
    "This is the last sentence."
)
# TODO 1 : appliquer le pipeline nlp au texte
doc = ...

# TODO 2 : parcourir les phrases détectées et les afficher
for ...:
    ...

Sentence 1: This is the first sentence.
Sentence 2: This is another sentence.
Sentence 3: This is the last sentence.


**À retenir :** `nlp(text)` crée un objet `Doc`. L’attribut `doc.sents` permet d’itérer sur les segments reconnus comme phrases.

## 4. Examiner les objets `Span`

In [ ]:
sentences = list(doc.sents)

print("Number of sentences:", len(sentences))
print("Second sentence:", sentences[1].text)
print("Token interval [start, end):", sentences[1].start, sentences[1].end)
print("Object type:", type(sentences[1]))

In [4]:
# EXERCICE 2 — Examiner les objets Span
# TODO 1 : convertir les phrases en liste
sentences = ...

# TODO 2 : afficher le nombre de phrases
print(...)

# TODO 3 : afficher le texte de la deuxième phrase
print(...)

# TODO 4 : afficher les indices start et end de la deuxième phrase
print(...)

Number of sentences: 3
Second sentence: This is another sentence.
Token interval [start, end): 6 11
Object type: <class 'spacy.tokens.span.Span'>


Une phrase est représentée par un objet `Span`. Les indices `start` et `end` délimitent les tokens dans le document ; `end` est exclusif.

## 5. Repérer le début des phrases

In [5]:
# EXERCICE 3 — Repérer le début des phrases
text = "This is a sentence. That is another sentence. Here is a third."
doc = nlp(text)

# TODO : afficher chaque token et token.is_sent_start
for ...:
    ...

      This | is_sent_start=True
        is | is_sent_start=False
         a | is_sent_start=False
  sentence | is_sent_start=False
         . | is_sent_start=False
      That | is_sent_start=True
        is | is_sent_start=False
   another | is_sent_start=False
  sentence | is_sent_start=False
         . | is_sent_start=False
      Here | is_sent_start=True
        is | is_sent_start=False
         a | is_sent_start=False
     third | is_sent_start=False
         . | is_sent_start=False


`token.is_sent_start` vaut `True` pour le premier token d’une phrase et `False` pour les autres tokens. Le premier token du document est généralement un début de phrase.

## 6. Pourquoi la segmentation peut être difficile

Un point peut terminer une phrase, mais il peut aussi apparaître dans une abréviation, une adresse web ou un nombre décimal. Les modèles linguistiques utilisent le contexte pour mieux décider.

In [6]:
# EXERCICE 4 — Observer les cas difficiles
examples = [
    "Hello Mr. Smith, how are you today?",
    "The value is 3.26 dollars. The meeting is on Monday.",
    "Visit example.com for more information."
]
# TODO : afficher le texte original puis chaque phrase détectée
for text in ...:
    print("\\nTEXT:", ...)
    doc = ...
    for sent in ...:
        print(...)


TEXT: Hello Mr. Smith, how are you today?
 - Hello Mr. Smith, how are you today?

TEXT: The value is 3.26 dollars. The meeting is on Monday.
 - The value is 3.26 dollars.
 - The meeting is on Monday.

TEXT: Visit example.com for more information.
 - Visit example.com for more information.


## 7. Segmentation personnalisée avec spaCy

Il est possible d’ajouter une règle métier. Exemple : considérer un point-virgule (`;`) comme une frontière de phrase. Le composant ci-dessous est ajouté **avant** l’analyseur syntaxique afin que celui-ci puisse tenir compte de la frontière.

In [7]:
# EXERCICE 5 — Ajouter une règle personnalisée
from spacy.language import Language

@Language.component("semicolon_sentence_boundary")
def semicolon_sentence_boundary(doc):
    # TODO : parcourir les tokens sauf le dernier
    for token in ...:
        # TODO : si le token est ";" marquer le token suivant
        # comme début de phrase
        if ...:
            ...
    return doc

if "semicolon_sentence_boundary" not in nlp.pipe_names:
    # TODO : ajouter le composant avant le parser
    ...

custom_text = "Management is doing things right; leadership is doing the right things."
custom_doc = nlp(custom_text)
# TODO : afficher les phrases
for sent in ...:
    ...

- Management is doing things right;
- leadership is doing the right things.


**Remarque :** les règles personnalisées doivent être évaluées sur des exemples représentatifs. Une frontière utile dans un cas peut produire une segmentation incorrecte dans un autre.

## 8. Utiliser NLTK

NLTK propose `sent_tokenize`, un outil pratique pour découper un texte en phrases. Il peut nécessiter le téléchargement de ressources de tokenisation.

In [ ]:
# EXERCICE 6 — Utiliser NLTK
import nltk
from nltk.tokenize import sent_tokenize

for resource in ("punkt", "punkt_tab"):
    nltk.download(resource, quiet=True)

example_text = (
    "Hello Mr. Smith, how are you doing today? "
    "The weather is great, and Python is awesome. "
    "The sky is pinkish-blue. You shouldn't eat cardboard."
)
# TODO : segmenter avec sent_tokenize et afficher chaque phrase numérotée
for ..:
    ...

1. Hello Mr. Smith, how are you doing today?
2. The weather is great, and Python is awesome.
3. The sky is pinkish-blue.
4. You shouldn't eat cardboard.


## 9. Comparer spaCy et NLTK

In [ ]:
# EXERCICE 7 — Comparer spaCy et NLTK
text = (
    "Dr. Smith arrived at 10.30 a.m. "
    "He gave a short talk. Everyone enjoyed it!"
)
print("spaCy:")
# TODO : segmenter avec spaCy et afficher chaque phrase
for ...:
    ...

print("\\nNLTK:")
# TODO : segmenter avec NLTK et afficher chaque phrase
for ...:
    ...

Les deux bibliothèques peuvent produire des résultats différents selon le texte, les modèles et les ressources utilisés. Pour un projet réel, examinez les cas limites de votre corpus.

## 10. Texte arabe : points d’attention

Le modèle anglais `en_core_web_sm` n’est pas adapté à l’analyse linguistique de l’arabe. Pour illustrer une segmentation simple, `spacy.blank('ar')` fournit un tokenizer arabe sans analyseur syntaxique. Les frontières de phrases doivent être ajoutées explicitement, par exemple avec `sentencizer`.

In [ ]:
# EXERCICE 8 — Segmenter un texte arabe
arabic_nlp = spacy.blank("ar")
# TODO : ajouter le composant sentencizer
...

arabic_text = "هذه هي الجملة الأولى. هذه هي الجملة الثانية؟ وهذه هي الجملة الثالثة!"
arabic_doc = arabic_nlp(arabic_text)
# TODO : afficher les phrases arabes avec leur numéro
for ...:
    ...

Cette démonstration dépend de la ponctuation. Pour une analyse morphologique ou syntaxique de l’arabe, choisissez un modèle et des ressources adaptés à la variété linguistique visée.

## 11. Découpage simple avec `str.split()`

La méthode Python `split()` sépare généralement sur les espaces. Elle ne tient pas compte de la structure linguistique et conserve souvent la ponctuation attachée aux mots.

In [ ]:
# EXERCICE 9 — Comparer avec str.split()
text = "Sentence one. Sentence two! Is this sentence three?"
# TODO : utiliser split() et afficher le résultat
...

## 12. Exercices

1. Écrivez un texte contenant au moins quatre phrases et affichez chaque phrase avec `spaCy`.
2. Comparez les résultats de `spaCy` et `NLTK` sur une phrase contenant une abréviation.
3. Testez le texte `I met Prof. Mohamed yesterday. We discussed NLP.` et observez la segmentation.
4. Segmentez un court texte arabe avec `spacy.blank('ar')` et `sentencizer`.
5. Expliquez pourquoi `str.split()` n’est pas une méthode fiable pour segmenter les phrases.

## 13. Résumé

- La segmentation des phrases identifie les unités phrastiques d’un texte.
- `spaCy` utilise ici un modèle anglais avec analyseur syntaxique ; `doc.sents` renvoie les phrases.
- `NLTK.sent_tokenize` propose une autre approche de segmentation.
- Les abréviations, nombres, URL et langues différentes constituent des cas particuliers.
- Les règles personnalisées doivent être testées et adaptées au corpus.

## 14. Bilan individuel — sans regarder de solution

1. Quelle est la différence entre un token et une phrase ?
2. À quoi sert `doc.sents` dans spaCy ?
3. Que représente un objet `Span` ?
4. Pourquoi un point ne marque-t-il pas toujours la fin d'une phrase ?
5. Quelles différences observez-vous entre spaCy et NLTK ?
6. Pourquoi `str.split()` ne suffit-il pas pour segmenter un texte ?
7. Quel est le rôle de `sentencizer` dans l'exemple arabe ?

### Auto-évaluation

- [ ] Je sais appliquer `nlp(text)` à un texte.
- [ ] Je sais parcourir et afficher les phrases.
- [ ] Je sais utiliser `sent_tokenize` de NLTK.
- [ ] Je sais expliquer une difficulté de segmentation.
- [ ] Je sais comparer deux méthodes de segmentation.